# Columnar analysis and Awkward Array

<br><br><br>

## Introduction

<b>"Columnar" is a term that tends to get overloaded, and I’ve used it in two distinct ways:</b>

* <b>Data layout:</b> organizing data in memory or on disk to enable faster, selective readout. (In fact, some types of TTree data have been stored this way since 1995.)
* <b>Array-oriented computation:</b> performing operations directly on entire arrays of data, rather than processing one value at a time in a loop. In other words — _no for loops_!

Of these, <b>only the second meaning directly affects physicists writing analysis code</b>.

That’s why I prefer to call it <b>"array-oriented programming"</b> — it’s a programming paradigm, much like "imperative" or "functional," describing _how you think about and organize your code_.

In [1]:
from IPython.display import HTML
HTML("""
<div style="width:1600px; height:600px; overflow:hidden;">
  <iframe src="./img/array_oriented_programming_timeline.html"
          style="width:1600px; height:1000px;
                 transform:scale(0.8); transform-origin:top left;"
          frameborder="0">
  </iframe>
</div>
""")

<br><br><br>

### Easy example of imperative, functional, and array-oriented

Compute the square of every element in a list/array.

<br><br><br>

#### Imperative

In [ ]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = []
for x in original:
    result.append(x**2)

result

<br><br><br>

#### Functional

In [ ]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = list(map(lambda x: x**2, original))

result

Functional programming with `map` isn't common in Python, but list comprehensions are pretty close to the "spirit" of functional programming:

In [ ]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = [x**2 for x in original]

result

<br><br><br>

#### Array-oriented

In [ ]:
import numpy as np

In [ ]:
original = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

result = original**2

result

<br><br><br>

### Hard example of imperative, functional, and array-oriented

Compute gravitational forces among $n$ particles in 3 dimensions.

#### Newton’s Law of Universal Gravitation

The gravitational force between two point masses $m_1$ and $m_2$ separated by a distance $r$ is given by:

$$
\vec{F} = G \frac{m_1 m_2}{r^2} \hat{r}
$$

Where:

* $\vec{F}$ is the gravitational force vector (Newtons)
* $G$ is the gravitational constant
* $m_1, m_2$ are the masses of the two objects
* $r$ is the distance between their centers
* $\hat{r}$ is the unit vector pointing from one mass to the other

In component form for positions $\vec{x}_i$ and $\vec{x}_j$:

$$
\vec{F}_{ij} = G \frac{m_i m_j}{|\vec{r}_{ij}|^3} \vec{r}_{ij}
$$

with

$$
\vec{r}_{ij} = \vec{x}_j - \vec{x}_i
$$


<br><br><br>

In [ ]:
m = np.array([100, 1, 1])   # sun and a double-planet (a 3-body problem)

# initial position (x) and momentum (p)
x = np.array([[0, 0, 0], [0, 0.9, 0], [0, 1.1, 0]])
p = np.array([[0, 0, 0], [-13, 0, 0], [-10, 0, 0]])

G = 1

<br><br><br>

#### Imperative

In [ ]:
def imperative_forces(m, x, p):
    # Initialize total force array with zeros, same shape as positions array x
    total_force = np.zeros_like(x)

    # Loop over each unique pair of particles (i, j) where i < j
    for i in range(len(x)):
        for j in range(i + 1, len(x)):
            
            # Extract masses of the two particles
            mi, mj = m[i], m[j]
            
            # Extract positions of the two particles
            xi, xj = x[i], x[j]
            
            # Compute displacement vector from particle i to particle j
            displacement = [
                xj[0] - xi[0],
                xj[1] - xi[1],
                xj[2] - xi[2],
            ]
            
            # Compute Euclidean distance between the two particles
            distance = np.sqrt(displacement[0]**2 + displacement[1]**2 + displacement[2]**2)
            
            # Compute unit vector (direction) from particle i to particle j
            direction = [
                displacement[0] / distance,
                displacement[1] / distance,
                displacement[2] / distance,
            ]

            # Calculate gravitational force magnitude and vector components
            # Newton's law of gravitation: F = G * m_i * m_j / r^2 * direction
            force = [
                G * mi * mj * direction[0] / distance**2,
                G * mi * mj * direction[1] / distance**2,
                G * mi * mj * direction[2] / distance**2,
            ]

            # Add the force to particle i (towards particle j)
            total_force[i, 0] += force[0]
            total_force[i, 1] += force[1]
            total_force[i, 2] += force[2]

            # Subtract the force from particle j (equal and opposite reaction)
            total_force[j, 0] += -force[0]
            total_force[j, 1] += -force[1]
            total_force[j, 2] += -force[2]

    # Return the array of total forces on each particle
    return total_force

<br><br><br>

#### Functional

In [ ]:
from functools import reduce
from itertools import combinations

In [ ]:
def functional_forces(m, x, p):
    def negate(vector):
        # Negate each component of the vector
        return [-a for a in vector]

    def add(*vectors):
        # Component-wise sum of multiple vectors
        return [reduce(lambda a, b: a + b, components) for components in zip(*vectors)]

    def subtract(vectorA, vectorB):
        # Vector subtraction: vectorA - vectorB
        return add(vectorA, negate(vectorB))

    def magnitude(vector):
        # Euclidean norm of the vector
        return np.sqrt(reduce(lambda a, b: a + b, map(lambda a: a**2, vector)))

    def force(mi, mj, xi, xj, pi, pj):
        # Calculate gravitational force vector exerted by j on i
        displacement = subtract(xj, xi)  # displacement vector from i to j
        distance = magnitude(displacement)
        direction = [a / distance for a in displacement]
        return [G * mi * mj * a / distance**2 for a in direction]

    # Generate all unique pairs of particles and their forces
    pairwise_forces = [
        ((i, j), force(mi, mj, xi, xj, pi, pj))
        for ((i, (mi, xi, pi)), (j, (mj, xj, pj))) in combinations(enumerate(zip(m, x, p)), 2)
    ]

    def partial_forces(pairwise_forces, i):
        # Collect all forces acting on particle i.
        # force(mi, mj, xi, xj) computes displacement xj-xi, so the result
        # points from i toward j — i.e. it is the force ON particle i.
        #   when i is first in the pair (a==i): force is already the force on i → keep it
        #   when i is second in the pair (b==i): force was computed for the other particle → negate it
        return (
            [force         for ((a, b), force) in pairwise_forces if a == i] +  # force ON i: keep
            [negate(force) for ((a, b), force) in pairwise_forces if b == i]    # force on other: flip
        )

    # Sum all partial forces for each particle and return as numpy array
    return np.array([add(*partial_forces(pairwise_forces, i)) for i in range(len(m))])

<br><br><br>

#### Array-oriented

In [ ]:
def array_forces(m, x, p):
    # Get the indices of the upper triangle of the pairwise interaction matrix (i < j)
    # This ensures each pair is considered only once (no double counting)
    i, j = np.triu_indices(len(x), k=1)

    # Calculate displacement vectors between pairs of particles (j - i)
    pw_displacement = x[j] - x[i]

    # Compute the Euclidean distance between particle pairs
    # Sum of squared components followed by square root gives distance
    pw_distance = np.sqrt(np.sum(pw_displacement**2, axis=-1))

    # Calculate the unit direction vectors from particle i to particle j
    # Divide displacement vector by distance to normalize
    pw_direction = pw_displacement / pw_distance[:, np.newaxis]

    # Calculate the gravitational force vector for each pair using Newton's law of gravitation:
    # F = G * m_i * m_j / r^2 * direction
    # The [:, np.newaxis] adds a new axis for broadcasting over x,y,z components
    pw_force = G * m[i, np.newaxis] * m[j, np.newaxis] * pw_direction / pw_distance[:, np.newaxis]**2

    # Initialize array to accumulate total forces on each particle (same shape as x)
    total_force = np.zeros_like(x)

    # Add force vectors to particle i's total force
    np.add.at(total_force, i, pw_force)

    # Subtract equal and opposite force vectors from particle j's total force
    # This enforces Newton's third law: action = -reaction
    np.add.at(total_force, j, -pw_force)

    # Return the total forces acting on all particles
    return total_force

<br><br><br>

In [ ]:
imperative_forces(m, x, p)

In [ ]:
functional_forces(m, x, p)

In [ ]:
array_forces(m, x, p)

<br><br><br>

#### Let's see it!

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [ ]:
def array_step(m, x, p, dt):
    """Leapfrog (Störmer-Verlet) integrator. Modifies x and p in place."""
    # Half-kick / full-drift / half-kick (numerically stable symplectic integrator)
    p += array_forces(m, x, p) * (dt/2)    # half kick
    x += p * dt / m[:, np.newaxis]         # full drift
    p += array_forces(m, x, p) * (dt/2)    # half kick

In [ ]:
def plot(m, x, p, dt, num_frames=100, steps_per_frame=10):
    num_particles = len(m)  # Number of particles

    # Allocate an empty array to store particle positions over time
    # Shape: (num_frames, num_particles, 2) for 2D positions (x, y)
    history = np.empty((num_frames, num_particles, 2))
    
    # Simulate and record positions at each frame
    for i in range(num_frames):
        # Save current x positions for all particles
        history[i, :, 0] = x[:, 0]
        # Save current y positions for all particles
        history[i, :, 1] = x[:, 1]

        # Advance simulation by 'steps_per_frame' small steps between frames
        for _ in range(steps_per_frame):
            array_step(m, x, p, dt)  # Update positions and momenta in place

    # Set up the plotting figure and axis with fixed size
    fig, ax = plt.subplots(figsize=(5, 5))

    lines = []
    # Initialize empty trajectory lines for each particle
    for j in range(num_particles):
        # Plot initial line segment (single point for now) for each particle
        line, = ax.plot(history[:1, j, 0], history[:1, j, 1])
        lines.append(line)

    # Plot particles as scatter points at initial positions
    dots = ax.scatter(history[0, :, 0], history[0, :, 1])

    # Fix axis limits to keep view constant during animation
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)

    # Define update function called at each animation frame
    def update(i):
        # Update trajectory lines to show all positions up to frame i
        for j, line in enumerate(lines):
            line.set_xdata(history[:i, j, 0])
            line.set_ydata(history[:i, j, 1])
        # Update scatter plot positions to frame i positions
        dots.set_offsets(history[i, :, :])
        # Return updated artists for blitting (efficient redraw)
        return [*lines, dots]

    # Create animation object
    ani = animation.FuncAnimation(
        fig=fig,          # Figure to animate
        func=update,      # Update function called each frame
        frames=num_frames, # Total number of frames
        interval=50,      # Delay between frames in milliseconds
        blit=True         # Use blitting for performance
    )
    
    # Convert animation to HTML5 format for display (e.g., in Jupyter)
    out = HTML(ani.to_jshtml())
    
    # Close figure to avoid duplicate static plot display
    plt.close()
    
    # Return HTML animation object for display
    return out

In [ ]:
m = np.array([100, 1, 1], np.float64)
x = np.array([[0, 0, 0], [0, 0.9, 0], [0, 1.1, 0]], np.float64)
p = np.array([[0, 0, 0], [-13, 0, 0], [-10, 0, 0]], np.float64)

plot(m, x, p, dt=0.001)

In [ ]:
# Define momentum components (velocity * mass) for particles 0 and 1
a = 0.347111
b = 0.532728

# Mass array for 3 particles, all unit mass
m = np.array([1, 1, 1], np.float64)

# Initial positions of particles in 3D space
# Particle 0 at (-1, 0, 0)
# Particle 1 at (1, 0, 0)
# Particle 2 at origin (0, 0, 0)
x = np.array([[-1, 0, 0], [1, 0, 0], [0, 0, 0]], np.float64)

# Initial momenta of particles (momentum = mass * velocity)
# Particles 0 and 1 have the same momentum vector (a, b, 0)
# Particle 2 has momentum exactly opposite to the sum of the other two,
# ensuring total momentum sums to zero (momentum conservation)
p = np.array([
    [a, b, 0],       # Particle 0
    [a, b, 0],       # Particle 1
    [-2 * a, -2 * b, 0]  # Particle 2
], np.float64)

# Run simulation and plot particle trajectories with timestep dt=0.01
plot(m, x, p, dt=0.01)

In [ ]:
# Create an array of masses for 25 particles, all with mass = 1
m = np.ones(25)

# Initialize positions of 25 particles randomly in 3D space
# Each coordinate is drawn from a normal distribution with mean=0 and std=1
x = np.random.normal(0, 1, (25, 3))

# Initialize momenta of 25 particles randomly in 3D
# Each component drawn from normal distribution with mean=0 and std=1
p = np.random.normal(0, 1, (25, 3))

# Run simulation and animate particle trajectories with a timestep of 0.0025
plot(m, x, p, dt=0.0025)

<br><br><br>

#### Let's time it!

In [ ]:
m = np.ones(500)
x = np.random.normal(0, 1, (500, 3))
p = np.random.normal(0, 1, (500, 3))

In [ ]:
%%timeit -n3 -r3

imperative_forces(m, x, p)

In [ ]:
%%timeit -n3 -r3

functional_forces(m, x, p)

In [ ]:
%%timeit -n3 -r3

array_forces(m, x, p)

In [ ]:
import dis

disassembled = dis.Bytecode(imperative_forces)
instructions = list(disassembled)
print(f"Number of bytecode instructions: {len(instructions)}")

In [ ]:
disassembled = dis.Bytecode(functional_forces)
instructions = list(disassembled)
print(f"Number of bytecode instructions: {len(instructions)}")

In [ ]:
disassembled = dis.Bytecode(array_forces)
instructions = list(disassembled)
print(f"Number of bytecode instructions: {len(instructions)}")

### Performance summary

Running the three implementations on 500 particles (`-n3 -r3`) typically shows results like:

| Implementation | Time | Bytecodes |
|---|---|---|
| `imperative_forces` | 251 ms ± 7 ms | 230 |
| `functional_forces` | 1.7 s ± 10 ms | 137 + call-stack overhead |
| `array_forces` | 9.89 ms ± 444 μs | 88 (executed once, not once per pair) |

The array-oriented version is **~25× faster than imperative** and **~170× faster than functional** — not because the math is different, but because Python's interpreter overhead applies to the whole array at once rather than to each of the 500 × 499 / 2 = 124,750 particle pairs.

The functional version is the *slowest* despite having fewer bytecodes, because Python's call-stack overhead from all the nested `lambda`/`map`/`reduce` calls adds up quickly.

<br><br><br>

In Python, array-oriented programming is a big advantage because it avoids Python's overhead (virtual machine, dynamic data types, garbage collection, etc).

* In `imperative_forces`, each of the 500 × 499 pairs of particles has to step through 30 lines of Python code (230 byte-code instructions).
* In `functional_forces`, each of the 500 × 499 pairs of particles has to step through functions that add up to 24 lines of Python code (137 byte-code instructions), plus call-stack overhead.
* `array_forces` has 10 lines of Python code (88 byte-code instructions), but the Python virtual machine steps through them only once, _not once per pair of particles_.

<br><br><br>

This is also relevant for GPU programming.

To get the most performance out of GPU programming frameworks like CUDA, you need to arrange the computation as array-at-a-time and think about vectorization.

<br><br><br>

### What if imperative code is easier to reason about?

Sometimes it is.

If it's easier to think about a problem imperatively, but the loop would iterate over some large number, just make sure Python isn't implementing the loop.

* Just-In-Time (JIT) compile it with [Numba](https://numba.pydata.org/).
* Use ROOT's [RDataFrame](https://root.cern/doc/master/classROOT_1_1RDataFrame.html) to compile and run C++ over ROOT data.
* Use [cppyy](https://cppyy.readthedocs.io/) to compile and run C++ over arbitrary data.
* Use [Julia](https://julialang.org/).
* Use [pybind11](https://pybind11.readthedocs.io/) to compile a Python extension in C++ or [PyO3](https://pyo3.rs/) to compile a Python extension in Rust.

All of these involve more set-up time to get started than array-oriented programming, but may be easier to deal with in the long run, depending on the problem.

<br><br><br>

## This tutorial

This tutorial will be about array-oriented programming in Python.

Exploring and analyzing data in an array-oriented way is a useful skill.

It's organized as a set of puzzles that we'll solve together. Open [student.ipynb](student.ipynb) now.

<br><br><br>

## Regular multi-dimensional arrays

In a <b>3D NumPy array</b>, the `axis` parameter specifies <b>which dimension an operation acts along</b>.

<img src="img/numpy_3d_axis.png" width="50%">

Since a 3D array has three dimensions, `axis` can take one of three possible values:

* `axis=0` → operates along the <b>first dimension</b>, moving <b>between 2D arrays</b> stacked along the "depth" of the 3D structure.
* `axis=1` → operates along the <b>second dimension</b>, moving <b>across rows within each 2D array</b> — like the "height" within a layer.
* `axis=2` → operates along the <b>third dimension</b>, moving <b>across columns within each row</b> — representing the "width" of the 3D array.

In short:

* `axis=0` → across different 2D slices
* `axis=1` → down rows in each 2D slice
* `axis=2` → across columns in each row

This `axis` argument lets you control the direction of operations like `sum()`, `mean()`, or `max()` in multi-dimensional arrays.

<br><br><br>

## Puzzles

### NumPy puzzle 1

Given a 3D array,

In [ ]:
import numpy as np

In [ ]:
array3d = np.arange(2 * 3 * 5).reshape(2, 3, 5)
array3d

you can select items

<img src="img/array3d-highlight1.svg" width="25%">

with

In [ ]:
array3d[:, 1:, 1:]

Write a slice the selects these elements:

<img src="img/array3d-highlight2.svg" width="25%">

<br><br><br>

#### Solution

##### Either keep the number of dimensions intact (the first dimension has length 1):

In [ ]:
array3d[:1, :, 2:]

##### or reduce that dimension:

In [ ]:
array3d[0, :, 2:]

These are different answers to the question, but both are fine.

<br><br><br>

### NumPy puzzle 2

Compute the size of the spaces between consecutive elements in the following array.

In [ ]:
array = np.array([1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8, 9.9])
array

**Hint:**

<img src="img/flat-operation.svg" width="40%"><img src="img/shifted-operation.svg" width="40%">

<br><br><br>

#### Solution

This involves either slicing and applying a universal function across all elements, showing how NumPy's elementary operations can be combined to get functionality that may not be explicitly compiled into the library or using a NumPy's diff function.

In [ ]:
array[1:] - array[:-1]

In [ ]:
np.diff(array)

<br><br><br>

### NumPy puzzle 3

Compute the length of this curve.

<img src="img/length-by-segment.svg" width="50%">

In [ ]:
t = np.linspace(0, 2*np.pi, 10000)
x = np.sin(3*t)
y = np.sin(4*t)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(x, y);

<br><br><br>

#### Solution

This calculation of path length involves slicing, universal functions, and reduction.

In [ ]:
# axis=0 is the only axis of a 1D array; it can be omitted, but is shown for clarity
np.sum(np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2), axis=0)

<br><br><br>

### NumPy puzzle 4

Scale this image down by a factor of 64 on both sides, using only [np.reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html), [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html), and [np.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html).

In [ ]:
import matplotlib.image

In [ ]:
image = matplotlib.image.imread("data/sun-shines-in-CMS.jpg")
plt.imshow(image);

The current shape is

In [ ]:
image.shape

1920 rows, 2560 columns, and the third axis is for (red, green, blue), all `np.uint8`.

Your strategy should be to reshape the array, such that the dimension of length `1920` becomes two new dimensions of length `1920 // 64` and `64` and the dimension of length `2560` becomes two new dimensions of length `2560 // 64` and `64`. Then average over each of the dimensions of length `64`.

The shape should change as

$$\left(1920, 2560, 3\right) \to \left(\frac{1920}{64}, 64, \frac{2560}{64}, 64, 3\right) \to \left(\frac{1920}{64}, \frac{2560}{64}, 3\right)$$

and then you need to turn the floating-point dtype back into unsigned 8-bit integers with [np.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html).

<br><br><br>

#### Solution

In NumPy, the shapes of arrays are fluid. An array can be reshaped from 3 dimensional to 5 dimensional as long as the number of items is unchanged.

Reshaping provides new axes for reducers like [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) to apply.

Be sure to average over axis 3 before axis 1, since reduction removes an axis, changing the numbering of later axes.


In [ ]:
h, w, c = image.shape  # use shape so the solution works for any image

resampled = np.mean(
    np.mean(
        image.reshape(h // 64, 64, w // 64, 64, c),
        axis=3,
    ),
    axis=1,
)


Or alternatively, use negative axis numbers and do it in the other order.

In [ ]:
# Using negative axis indices — order doesn't matter because negative indices
# are unaffected by earlier reductions removing axes.
resampled = np.mean(
    np.mean(
        image.reshape(h // 64, 64, w // 64, 64, c),
        axis=-4,
    ),
    axis=-2,
)


Or as a third option, use `keepdims=True` to keep the axis numbering from changing and only remove the length 1 dimensions after all reductions are done.

In [ ]:
# Using keepdims=True so axis numbers don't shift after each reduction.
resampled = np.mean(
    np.mean(
        image.reshape(h // 64, 64, w // 64, 64, c),
        axis=1,
        keepdims=True,
    ),
    axis=3,
    keepdims=True,
).reshape(h // 64, w // 64, c)


In [ ]:
plt.imshow(resampled.astype(np.uint8));

<br><br><br>

### NumPy puzzle 5

Let's interpret some raw ROOT data!

In [ ]:
import zlib

In [ ]:
with open("data/SMHiggsToZZTo4L.root", "rb") as file:
    file.seek(42104123)
    compressed_data = file.read(14718)
    uncompressed_data = uncompressed_data = zlib.decompress(compressed_data)
    array_of_uint8 = np.frombuffer(uncompressed_data, np.uint8, 12524)

In [ ]:
array_of_uint8

This should be 3131 floating-point muon $p_T$ values.

But if we view the data as `float32`, the orders of magnitude are all wrong:

In [ ]:
array_of_uint8.view(np.float32)

That's because the data are [big endian](https://en.wikipedia.org/wiki/Endianness) and this computer is little-endian.

NumPy has a way to view "wrong"-endian data, but operations on it aren't as efficient.

In [ ]:
# ">f4" = big-endian (">") 32-bit float ("f4") — NumPy dtype string notation
array_of_uint8.view(">f4")

To fix the endianness, you'll need to reverse the order of bytes _in groups of 4_.

<img src="img/big-little-endian.svg" width="75%">

Fix the endianness with only [np.reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) and a slice.

<br><br><br>

#### Solution

This is similar to the image-scaling puzzle: we use short-lived axes to be able to express an operation, but instead of reduction, it's a direction-reversing slice. Without reshaping, we wouldn't be able to express "groups of 4."

In [ ]:
byte_swapped = array_of_uint8.reshape(-1, 4)[:, ::-1].reshape(-1)

In [ ]:
byte_swapped.view(np.float32)

<br><br><br>

## Introduction to ragged arrays

In [ ]:
import awkward as ak

In [ ]:
array2d = np.array([
    [  1,   2,   3,   4],
    [ 10,  20,  30,  40],
    [100, 200, 300, 400],
])
array2d

In [ ]:
ragged = ak.Array([
    [   1,    2,    3,   4],
    [  10, None,   30     ],
    [ 100,  200           ],
])
ragged

Some operations, like summation along an axis, need to be generalized.

<img src="img/example-reducer-2d.svg" width="50%"><img src="img/example-reducer-ragged.svg" width="50%">

In [ ]:
np.sum(array2d, axis=0)

In [ ]:
np.sum(array2d, axis=1)

In [ ]:
ak.sum(ragged, axis=0)

In [ ]:
ak.sum(ragged, axis=1)

Particle physics data naturally comes in ragged arrays.

In [ ]:
import uproot

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root") as file:
    dataset = file["Events"].arrays()

In [ ]:
dataset

In [ ]:
dataset.Electron_pt

Almost always, `axis=0` is the _events_ axis (collision events), and `axis=1` is the _particle_ axis (electrons, muons, jets...).

Let's make this easier to use by defining:

In [ ]:
EVENT = 0
PARTICLE = 1

In [ ]:
ak.num(dataset, axis=EVENT)

The number of events is 299973. (See documentation for [ak.num](https://awkward-array.org/doc/main/reference/generated/ak.num.html).)

In [ ]:
ak.num(dataset.Electron_pt, axis=PARTICLE)

The number of electrons is 0 (in the first event), 4 (in the second event), 2 (in the third event), ... 2 (in the last event).

The following computes the average electron $p_T$ in each event:

In [ ]:
np.mean(dataset.Electron_pt, axis=PARTICLE)

**Question:** what is the following? (What does the calculation mean?)

In [ ]:
np.mean(dataset.Electron_pt, axis=EVENT)

<br><br><br>

## Awkward puzzles

### Awkward puzzle 1

Are the muon $p_T$ values sorted within each event? That is, is it true that

```python
dataset.Muon_pt[i, j] > dataset.Muon_pt[i, j + 1]
```

for all `i` and `j`? Perform a calculation that proves it.

<details>
    <summary style="font-weight: bold;">Hint!</summary>

<p>Think about NumPy puzzle 2.</p>

</details>

<br><br><br>

#### Solution

In NumPy puzzle 2, you subtracted each element from its neighbor by slicing array[1:] (drops the first) and array[:-1] (drops the last) before subtracting.

Here, you can do something similar, but it has to apply to the particles axis, not the events axis, so


In [ ]:
dataset.Muon_pt[:, :-1] > dataset.Muon_pt[:, 1:]

(either subtract and compare with zero or just compare one to the next).

We want to know if it's always true that `dataset.Muon_pt[:, :-1] > dataset.Muon_pt[:, 1:]`. We can see by eye that there are some `False` values above, but we can clinch it by

In [ ]:
np.all(dataset.Muon_pt[:, :-1] > dataset.Muon_pt[:, 1:])

It is not true that the muon $p_T$ values are sorted within each event.

The consequence, for this dataset, is that `dataset.Muon_pt[:, 0]` is not the leading $p_T$ and `0` and `1` are not necessarily the two most relevant muons.

<br><br><br>

### Awkward puzzle 2

Just as with NumPy arrays, Awkward Arrays can be filtered by slicing them by a boolean array of the same shape.

In [ ]:
boolean_mask = dataset.Muon_pt > 50
boolean_mask

In [ ]:
dataset.Muon_pt[boolean_mask]

Notice that this has filtered _particles_, rather than _events_. That's because `dataset.Muon_pt` and `boolean_mask` are both ragged (with the same numbers of elements in each list).

Suppose you want to select "any event with a muon whose $p_T > 50$ GeV"? Here are some potentially useful functions (there are several ways to do it):

* [ak.any](https://awkward-array.org/doc/main/reference/generated/ak.any.html)
* [ak.max](https://awkward-array.org/doc/main/reference/generated/ak.max.html)
* [ak.num](https://awkward-array.org/doc/main/reference/generated/ak.num.html)
* [ak.count_nonzero](https://awkward-array.org/doc/main/reference/generated/ak.count_nonzero.html) (or [ak.sum](https://awkward-array.org/doc/main/reference/generated/ak.sum.html))

<br><br><br>

#### Solution 1

Take an event if "any muon $p_T > 50$ GeV (considering all particles)."

In [ ]:
boolean_mask_1d = ak.any(dataset.Muon_pt > 50, axis=PARTICLE)
boolean_mask_1d

#### Solution 2

Take an event if "the maximum muon $p_T$ (over all particles) $> 50$ GeV."

In [ ]:
boolean_mask_1d = ak.max(dataset.Muon_pt, axis=PARTICLE) > 50
boolean_mask_1d

Some of these boolean values are missing because there is no maximum of an empty list, and that missing value (`None`) results in a missing conclusion about whether it is greater than 50 GeV or not.

The missing values can be converted into `False` with [ak.fill_none](https://awkward-array.org/doc/main/reference/generated/ak.fill_none.html)

In [ ]:
boolean_mask_1d = ak.fill_none(ak.max(dataset.Muon_pt, axis=PARTICLE) > 50, False)
boolean_mask_1d

#### Solution 3

Take an event if "the number of muons that have $p_T > 50$ GeV is $\ge 1$."

In [ ]:
boolean_mask_1d = ak.num(dataset.Muon_pt[dataset.Muon_pt > 50]) >= 1
boolean_mask_1d

#### Solution 4

Take an event if "the number of muons in which '$p_T > 50$ GeV' is true (not zero) $\ge 1$."

In [ ]:
boolean_mask_1d = ak.count_nonzero(dataset.Muon_pt > 50, axis=PARTICLE) >= 1
boolean_mask_1d

#### Solution 5

Take an event if "the sum of 'muon $p_T > 50$ GeV' (false = 0, true = 1) over particles is $\ge 1$."

In [ ]:
boolean_mask_1d = ak.sum(dataset.Muon_pt > 50, axis=PARTICLE) >= 1
boolean_mask_1d

#### Application

Apply it to the `dataset` as a slice. You get whole events (all fields) back.

In [ ]:
dataset[boolean_mask_1d]

Notice that the total number of events is lower, which is not the case for a particle cut.

<br><br><br>

### Awkward puzzle 3

Also like NumPy, arrays of booleans can be combined by logical operators `&` (and), `|` (or), `~` (not).

Be careful:

* The operators aren't `and`, `or`, `not`: the way these have been defined by Python, they can't apply to arrays.
* The operators aren't `&&`, `||`, `!`: that's C++.
* Comparisons have to be grouped with parentheses:

In [ ]:
x = 0

In [ ]:
x > -10 & x < 10

In [ ]:
(x > -10) & (x < 10)

<img src="img/bitwise-operator-parentheses.svg" width="75%">

Make a boolean mask for selecting events in which any electron $p_T > 10$ and any muon $p_T > 5$ GeV.

<br><br><br>

#### Solution

This answer can have the same variety as puzzle 2 because it asks for _any_ particle satisfying a condition, as above.

In [ ]:
ak.any(dataset.Electron_pt > 10, axis=PARTICLE) & ak.any(dataset.Muon_pt > 5, axis=PARTICLE)

<br><br><br>

### Awkward puzzle 4

Using [ak.num](https://awkward-array.org/doc/main/reference/generated/ak.num.html), select events with exactly two muons and make separate arrays (`mu1`, `mu2`) of those two muons.

<br><br><br>

#### Solution

You can use [ak.num](https://awkward-array.org/doc/main/reference/generated/ak.num.html) or the `nMuon` field of the `dataset`.

In [ ]:
selected = dataset[ak.num(dataset.Muon_pt) == 2]

Remember that the first axis is `EVENT` and we want all of them, so select `:`, then the second axis is `PARTICLE`.

In [ ]:
mu1 = selected.Muon_pt[:, 0]
mu2 = selected.Muon_pt[:, 1]

In [ ]:
mu1

In [ ]:
mu2

<br><br><br>

### Awkward puzzle 5

The `dataset` has separate ragged arrays for each particle attribute, such as `Electron_pt`, `Electron_phi`, `Electron_eta`, `Electron_mass`. It would be nicer to work with as arrays of electron _objects_ with Lorentz vector methods.

We can do that with [ak.zip](https://awkward-array.org/doc/main/reference/generated/ak.zip.html) and the [Vector](https://vector.readthedocs.io/) library.

In [ ]:
import vector
vector.register_awkward()

In [ ]:
electrons = ak.zip({
    "pt": dataset.Electron_pt,
    "phi": dataset.Electron_phi,
    "eta": dataset.Electron_eta,
    "mass": dataset.Electron_mass,
    "charge": dataset.Electron_charge,
}, with_name="Momentum4D")

In [ ]:
muons = ak.zip({
    "pt": dataset.Muon_pt,
    "phi": dataset.Muon_phi,
    "eta": dataset.Muon_eta,
    "mass": dataset.Muon_mass,
    "charge": dataset.Muon_charge,
}, with_name="Momentum4D")

`rapidity` is one of many methods in the [Vector](https://vector.readthedocs.io/) library.

In [ ]:
electrons.rapidity

The [Vector](https://vector.readthedocs.io/) library also defines some binary operators between particle objects, such as `+`.

In [ ]:
electrons + electrons

Now you're ready to compute dimuon masses.

1. Select events with exactly two muons and put each of those muons in separate arrays, as you did in puzzle 4 (but make them arrays of muon _objects_, not just muon $p_T$).
2. For each event, add those muon objects together with `+`.
3. Get the `mass` property from the Lorentz-added vectors.

It should look like

```
[87.1,
 90.5,
 89.2,
 18.8,
 4.59,
 37.2,
 90.6,
 8.91,
 18.1,
 27.5,
 ...,
 31.2,
 0.762,
 14.3,
 90.7,
 90,
 88.6,
 88.8,
 27.9,
 90.5]
---------------------
type: 85838 * float32
```

And maybe plot them.

In [ ]:
from hist import Hist

In [ ]:
def quickplot(num_bins, low, high, array, label=""):
    return Hist.new.Reg(num_bins, low, high, label=label).Double().fill(array)

In [ ]:
quickplot(100, 0, 100, ..., "mass (GeV)")

<br><br><br>

#### Solution

In [ ]:
selected_muons = muons[ak.num(muons) == 2]

In [ ]:
result = (selected_muons[:, 0] + selected_muons[:, 1]).mass
result

In [ ]:
quickplot(100, 0, 100, result, "mass (GeV)")

<br><br><br>

### Awkward puzzle 6

It's too restrictive to ask for events with exactly two muons, since spurious detector signals can show up as low-momentum fake particles.

In general, we want to test all _combinations_ of particles to see if they are consistent with an event topology.

To enable this, Awkward Array has functions for generating combinations of objects, [ak.cartesian](https://awkward-array.org/doc/main/reference/generated/ak.cartesian.html) and [ak.combinations](https://awkward-array.org/doc/main/reference/generated/ak.combinations.html).

<table>
    <tr style="background: white; text-align: center; font-size: 18pt">
        <td>ak.cartesian</td><td>ak.combinations</td>
    </tr>
    <tr style="background: white">
        <td><img src="img/cartoon-cartesian.png" width="300"></td>
        <td><img src="img/cartoon-combinations.png" width="300"></td>
    </tr>
</table>

<br><br><br>

In [ ]:
numbers = ak.Array([[1, 2, 3], [], [4]])
letters = ak.Array([["a", "b"], ["c"], ["d", "e"]])

ak.cartesian([numbers, letters], axis=1)

is equivalent to

In [ ]:
for numbers_list, letters_list in zip(numbers, letters):
    print("[", end="")
    for number in numbers_list:
        for letter in letters_list:
            print(f"({number}, {letter}), ", end="")
    print("]")

<br><br><br>

In [ ]:
numbers = ak.Array([[1.1, 2.2, 3.3, 4.4], [], [5.5, 6.6]])

ak.combinations(numbers, 2, axis=1)

is equivalent to

In [ ]:
for numbers_list in numbers:
    print("[", end="")
    for i in range(len(numbers_list)):
        for j in range(i + 1, len(numbers_list)):
            print(f"({numbers_list[i]}, {numbers_list[j]}), ", end="")
    print("]")

<br><br><br>

"For each event, construct all combinations of 2 muons."

In [ ]:
muon_pairs = ak.combinations(muons, 2, axis=PARTICLE)
muon_pairs

"Get the left and right halves of each pair as separate arrays," using [ak.unzip](https://awkward-array.org/doc/main/reference/generated/ak.unzip.html).

In [ ]:
# ak.unzip splits an array of tuples into separate arrays — one per field.
# Docs: https://awkward-array.org/doc/main/reference/generated/ak.unzip.html
mu1, mu2 = ak.unzip(muon_pairs)


(This could also be done with `mu1 = muon_pairs["0"]` and `mu2 = muon_pairs["1"]`.)

In [ ]:
mu1

In [ ]:
mu2

"Compute dimuon masses from these."

In [ ]:
dimuon_masses = (mu1 + mu2).mass
dimuon_masses

Now we plot it, using [ak.flatten](https://awkward-array.org/doc/main/reference/generated/ak.flatten.html) to ignore the boundaries between lists (turn the ragged array into a 1-dimensional array).

In [ ]:
quickplot(100, 0, 100, ak.flatten(dimuon_masses), label="mass (GeV)")

<br><br><br>

To ignore same-charge pairs, we could apply a cut to `mu1` and `mu2`,

In [ ]:
cut = mu1.charge != mu2.charge

quickplot(100, 0, 100, ak.flatten((mu1[cut] + mu2[cut]).mass), label="mass GeV")

but we could also avoid it by starting with separate `muon_plus` and `muon_minus` collections:

In [ ]:
muon_plus = muons[muons.charge > 0]
muon_minus = muons[muons.charge < 0]

Use these separate-charge collections, [ak.cartesian](https://awkward-array.org/doc/main/reference/generated/ak.cartesian.html), and [ak.unzip](https://awkward-array.org/doc/main/reference/generated/ak.unzip.html) to make the second plot above.

<br><br><br>

#### Solution

In [ ]:
mu1, mu2 = ak.unzip(ak.cartesian([muon_plus, muon_minus]))

quickplot(100, 0, 100, ak.flatten((mu1 + mu2).mass), label="mass GeV")

<br><br><br>

### Awkward puzzle 7

Using `muon_plus`, `muon_minus` and

In [ ]:
electron_plus = electrons[electrons.charge > 0]
electron_minus = electrons[electrons.charge < 0]

find quadruples of $e^+$, $e^-$, $\mu^+$, $\mu^-$ (without cuts) and plot a Higgs peak at 125 GeV. Remember to [ak.flatten](https://awkward-array.org/doc/main/reference/generated/ak.flatten.html) the ragged array of masses before plotting.

<img src="img/higgs-to-four-leptons-diagram.png" width="50%">

<br><br><br>

#### Solution 1

One way, by combining pairs of pairs:

In [ ]:
muon_pairs = ak.cartesian([muon_plus, muon_minus])
electron_pairs = ak.cartesian([electron_plus, electron_minus])

higgs_candidates = ak.cartesian([muon_pairs, electron_pairs])
higgs_candidates

These could be pulled apart with [ak.unzip](https://awkward-array.org/doc/main/reference/generated/ak.unzip.html), but because the tuples are nested within tuples, it's a little easier to access them by direct address.

`"0"` gets the first field (and `"1"` gets the second field) of _every tuple in an array_. This is very different from selecting `0` and `1` without quotes.

In [ ]:
mu1 = higgs_candidates["0"]["0"]
mu2 = higgs_candidates["0"]["1"]
e1 = higgs_candidates["1"]["0"]
e2 = higgs_candidates["1"]["1"]

In [ ]:
quickplot(150, 0, 150, ak.flatten((mu1 + mu2 + e1 + e2).mass))

<br><br><br>

#### Solution 2

But if we don't need to address the dimuons and dielectrons directly (to perform cuts on them), we could just use [ak.cartesian](https://awkward-array.org/doc/main/reference/generated/ak.cartesian.html) to make quadruples directly.

In [ ]:
quadruples = ak.cartesian([muon_plus, muon_minus, electron_plus, electron_minus])
quadruples

Doing it this way, they're not nested, and one [ak.unzip](https://awkward-array.org/doc/main/reference/generated/ak.unzip.html) will pull them apart.

<br><br><br>

### Awkward puzzle 8

Using `muon_plus` and `muon_minus`, find quadruples of $\mu^+$, $\mu^-$, $\mu^+$, $\mu^-$ (without cuts) and plot a Higgs peak.

Very important: don't include the same muon twice! An individual muon can't be the decay product of one Z _and_ the other Z.

<img src="img/higgs-to-four-leptons-diagram-2.png" width="50%">

<br><br><br>

#### Solution

In [ ]:
muon_pairs = ak.cartesian([muon_plus, muon_minus])

pairs_of_muon_pairs = ak.combinations(muon_pairs, 2)

mu1 = pairs_of_muon_pairs["0"]["0"]
mu2 = pairs_of_muon_pairs["0"]["1"]
mu3 = pairs_of_muon_pairs["1"]["0"]
mu4 = pairs_of_muon_pairs["1"]["1"]

In [ ]:
quickplot(150, 0, 150, ak.flatten((mu1 + mu2 + mu3 + mu4).mass))

<br><br><br>

### Awkward puzzle 9

Another common pattern is to

1. expand some data with combinatorics,
2. compute something on the pairs/triples/quadruples of particles,
3. perform a reduction over the computed quantity.

For example, if we need to match objects with $\Delta R$.

[ak.cartesian](https://awkward-array.org/doc/main/reference/generated/ak.cartesian.html) has a `nested` argument to ask for combinatorics to be grouped by one (or more) of the input arrays.

In [ ]:
numbers = ak.Array([[1, 2, 3], [], [4]])
letters = ak.Array([["a", "b"], ["c"], ["d", "e"]])

ak.cartesian([numbers, letters], axis=1)

In [ ]:
ak.cartesian([numbers, letters], axis=1, nested=True)

<br><br><br>

Although it isn't physically relevant in this sample, let's try computing $\Delta R$ between all electrons and muons.

In [ ]:
e_mu_pairs = ak.cartesian([electrons, muons], nested=True)

In [ ]:
e_grouped, mu_grouped = ak.unzip(e_mu_pairs)

In [ ]:
e_grouped

In [ ]:
mu_grouped

Notice the `299973 * var * var * Momentum4D`. We've turned two ragged arrays of particles into ragged-ragged arrays.

* `axis=0` ranges over events.
* `axis=1` ranges over electrons (the first of the electron-muon pairs).
* `axis=2` ranges over muons (the second of the electron-muon pairs).

Perhaps we should extend our set of named axes.

In [ ]:
EVENT = 0
PARTICLE = 1
PARTICLE2 = 2

Now when we use [Vector](https://vector.readthedocs.io/) to compute $\Delta R$, the $\Delta R$ values are also doubly-nested, one for each muon in lists for each electron.

In [ ]:
deltaR = e_grouped.deltaR(mu_grouped)
deltaR

Find the one muon that is closest in $\Delta R$ to each electron (or `None` if the event has no muons). Use [ak.argmin](https://awkward-array.org/doc/main/reference/generated/ak.argmin.html) with `keepdims=True`.

It should look like this:

```
[[],
 [None, None, None, None],
 [None, None],
 [{pt: 23.5, phi: -0.307, eta: -0.248, mass: 0.106, charge: -1}],
 [None, None, None, None],
 [{pt: 47, phi: -1.15, eta: -0.119, mass: 0.106, charge: 1}],
 [{pt: 4.45, phi: 1.12, eta: -0.986, mass: 0.106, charge: 1}],
 [None],
 [None, None, None, None],
 [],
 ...,
 [{pt: 37.2, phi: -0.875, eta: 1.1, mass: 0.106, charge: -1}, {...}],
 [{pt: 24, phi: 1.38, eta: 0.421, mass: 0.106, charge: -1}],
 [],
 [None, None, None],
 [],
 [{pt: 43.1, phi: 2.27, eta: -0.162, mass: 0.106, charge: -1}],
 [],
 [None, None],
 [None, None]]
---------------------------------------------------------------------
type: 299973 * var * ?Momentum4D[
    pt: float32,
    phi: float32,
    eta: float32,
    mass: float32,
    charge: int32
]
```

<br><br><br>

#### Solution

In [ ]:
EVENT = 0
PARTICLE = 1
PARTICLE2 = 2

In [ ]:
e_mu_pairs = ak.cartesian([electrons, muons], nested=True)
e_grouped, mu_grouped = ak.unzip(e_mu_pairs)
deltaR = e_grouped.deltaR(mu_grouped)

In [ ]:
# ak.argmin with keepdims=True keeps the PARTICLE2 axis (length 1) so the
# indexing into mu_grouped works. The final [:, :, 0] removes that length-1
# axis, giving one muon (or None) per electron per event.
mu_grouped[ak.argmin(deltaR, axis=PARTICLE2, keepdims=True)][:, :, 0]

<br><br><br>